# US Healthcare Data Breach Analysis (2015-2025)
**Author:** Olamide Bakare | Data Engineer & Data Governance Specialist

This notebook analyses healthcare data breaches reported to the US Department of Health and Human Services (HHS) Office for Civil Rights (OCR), exploring patterns in breach types, entity types, geographic distribution, reporting timelines, and data governance implications.

## Key Questions
1. How has the frequency and scale of healthcare breaches changed over time?
2. What types of breaches are most common, and how has this shifted?
3. Which states and entity types are most affected?
4. How quickly are breaches being reported, and what does this reveal about data governance maturity?
5. What are the data governance implications of these findings?

In [ ]:
# Import libraries
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

# Display settings
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 20)
plt.style.use('dark_background')

# Color palette
ACCENT = '#00e5a0'
ACCENT2 = '#00b8d4'
ACCENT3 = '#ff6b6b'
BG = '#0a0a0f'

print('Libraries loaded successfully')

## 1. Data Loading & Initial Exploration

In [ ]:
# Load the dataset
df = pd.read_csv('../data/healthcare_breaches.csv')

# Convert date columns
df['Breach_Date'] = pd.to_datetime(df['Breach_Date'], format='%m/%d/%Y')
df['Breach_Submission_Date'] = pd.to_datetime(df['Breach_Submission_Date'], format='%m/%d/%Y')

print(f'Dataset shape: {df.shape}')
print(f'Date range: {df["Breach_Date"].min().strftime("%Y-%m-%d")} to {df["Breach_Date"].max().strftime("%Y-%m-%d")}')
print(f'Total individuals affected: {df["Individuals_Affected"].sum():,.0f}')
print(f'\nColumn types:')
print(df.dtypes)

In [ ]:
# Data quality check - a critical data governance step
print('=== DATA QUALITY ASSESSMENT ===')
print(f'\nMissing values:')
print(df.isnull().sum())
print(f'\nDuplicate rows: {df.duplicated().sum()}')
print(f'\nUnique breach types: {df["Type_of_Breach"].nunique()}')
print(df['Type_of_Breach'].value_counts())
print(f'\nUnique entity types: {df["Covered_Entity_Type"].nunique()}')
print(df['Covered_Entity_Type'].value_counts())

## 2. Breach Frequency Analysis
### How has the number of healthcare breaches changed over time?

In [ ]:
# Breaches per year
fig, ax = plt.subplots(figsize=(12, 6), facecolor=BG)
ax.set_facecolor(BG)
yearly = df.groupby('Year').size()
bars = ax.bar(yearly.index, yearly.values, color=ACCENT, alpha=0.85, width=0.7)
for bar, val in zip(bars, yearly.values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 10, str(val),
            ha='center', va='bottom', fontsize=10, color='white', fontweight='bold')
ax.set_title('Healthcare Data Breaches Per Year (2015-2025)', fontsize=16, fontweight='bold', pad=20)
ax.set_xlabel('Year', fontsize=12, color='#8888a0')
ax.set_ylabel('Number of Breaches', fontsize=12, color='#8888a0')
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
plt.tight_layout()
plt.savefig('../visualisations/01_breaches_per_year.png', dpi=150, bbox_inches='tight', facecolor=BG)
plt.show()

print(f'\nTotal breaches 2015-2019: {yearly[2015:2019].sum()}')
print(f'Total breaches 2020-2025: {yearly[2020:2025].sum()}')
print(f'Increase: {((yearly[2020:2025].sum() / yearly[2015:2019].sum()) - 1) * 100:.1f}%')

### Individuals affected per year

In [ ]:
fig, ax = plt.subplots(figsize=(12, 6), facecolor=BG)
ax.set_facecolor(BG)
affected = df.groupby('Year')['Individuals_Affected'].sum() / 1e6
bars = ax.bar(affected.index, affected.values, color=ACCENT3, alpha=0.85, width=0.7)
for bar, val in zip(bars, affected.values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5, f'{val:.1f}M',
            ha='center', va='bottom', fontsize=10, color='white', fontweight='bold')
ax.set_title('Individuals Affected by Healthcare Breaches (Millions)', fontsize=16, fontweight='bold', pad=20)
ax.set_xlabel('Year', fontsize=12, color='#8888a0')
ax.set_ylabel('Individuals (Millions)', fontsize=12, color='#8888a0')
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
plt.tight_layout()
plt.savefig('../visualisations/02_individuals_affected.png', dpi=150, bbox_inches='tight', facecolor=BG)
plt.show()

## 3. Breach Type Analysis
### How has the nature of breaches evolved?

In [ ]:
# Hacking as percentage of all breaches over time
fig, ax = plt.subplots(figsize=(12, 6), facecolor=BG)
ax.set_facecolor(BG)
df['Is_Hacking'] = df['Type_of_Breach'] == 'Hacking/IT Incident'
hack_pct = df.groupby('Year')['Is_Hacking'].mean() * 100
ax.plot(hack_pct.index, hack_pct.values, color=ACCENT3, linewidth=3, marker='o', markersize=8)
ax.fill_between(hack_pct.index, hack_pct.values, alpha=0.15, color=ACCENT3)
for x, y in zip(hack_pct.index, hack_pct.values):
    ax.text(x, y + 2, f'{y:.0f}%', ha='center', fontsize=10, color='white', fontweight='bold')
ax.set_title('Hacking/IT Incidents as % of All Breaches', fontsize=16, fontweight='bold', pad=20)
ax.set_ylim(0, 100)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
plt.tight_layout()
plt.savefig('../visualisations/06_hacking_trend.png', dpi=150, bbox_inches='tight', facecolor=BG)
plt.show()

print('\n--- DATA GOVERNANCE INSIGHT ---')
print(f'In 2015, hacking accounted for {hack_pct[2015]:.0f}% of breaches.')
print(f'In 2025, hacking accounts for {hack_pct[2025]:.0f}% of breaches.')
print('This shift suggests that physical security controls (preventing theft, loss, improper disposal)')
print('have improved, but digital data governance (access controls, encryption, monitoring) has not kept pace.')

## 4. Geographic Analysis
### Which states are most affected?

In [ ]:
fig, ax = plt.subplots(figsize=(12, 6), facecolor=BG)
ax.set_facecolor(BG)
top_states = df['State'].value_counts().head(10)
bars = ax.barh(range(len(top_states)), top_states.values, color=ACCENT2, alpha=0.85)
ax.set_yticks(range(len(top_states)))
ax.set_yticklabels(top_states.index, fontsize=11)
for bar, val in zip(bars, top_states.values):
    ax.text(bar.get_width() + 5, bar.get_y() + bar.get_height()/2, str(val),
            ha='left', va='center', fontsize=10, color='white', fontweight='bold')
ax.set_title('Top 10 States by Number of Healthcare Breaches', fontsize=16, fontweight='bold', pad=20)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.invert_yaxis()
plt.tight_layout()
plt.savefig('../visualisations/04_top_states.png', dpi=150, bbox_inches='tight', facecolor=BG)
plt.show()

## 5. Reporting Timeline Analysis
### How quickly are breaches being reported?
This is a critical data governance metric. HIPAA requires breaches affecting 500+ individuals to be reported within 60 days.

In [ ]:
fig, ax = plt.subplots(figsize=(12, 6), facecolor=BG)
ax.set_facecolor(BG)
ax.hist(df['Days_to_Report'], bins=40, color=ACCENT, alpha=0.75)
median_delay = df['Days_to_Report'].median()
ax.axvline(60, color='#ffc107', linewidth=2, linestyle='--', label='HIPAA 60-day requirement')
ax.axvline(median_delay, color=ACCENT3, linewidth=2, linestyle='--', label=f'Median: {median_delay:.0f} days')
ax.set_title('Distribution of Days Between Breach and Report', fontsize=16, fontweight='bold', pad=20)
ax.set_xlabel('Days to Report', fontsize=12, color='#8888a0')
ax.set_ylabel('Number of Breaches', fontsize=12, color='#8888a0')
ax.legend(fontsize=11, framealpha=0.3)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
plt.tight_layout()
plt.savefig('../visualisations/07_reporting_delay.png', dpi=150, bbox_inches='tight', facecolor=BG)
plt.show()

over_60 = (df['Days_to_Report'] > 60).mean() * 100
print(f'\n--- DATA GOVERNANCE INSIGHT ---')
print(f'Median reporting delay: {median_delay:.0f} days')
print(f'{over_60:.1f}% of breaches are reported after the 60-day HIPAA requirement.')
print('This suggests significant gaps in breach detection and incident response governance.')

## 6. Entity Type Analysis

In [ ]:
fig, ax = plt.subplots(figsize=(8, 8), facecolor=BG)
ax.set_facecolor(BG)
entity_counts = df['Covered_Entity_Type'].value_counts()
colors_pie = [ACCENT, ACCENT2, ACCENT3, '#ffc107']
wedges, texts, autotexts = ax.pie(entity_counts.values, labels=entity_counts.index,
    autopct='%1.1f%%', colors=colors_pie, startangle=90,
    textprops={'color': 'white', 'fontsize': 11},
    pctdistance=0.8, wedgeprops=dict(width=0.5, edgecolor=BG))
for t in autotexts:
    t.set_fontweight('bold')
ax.set_title('Breaches by Entity Type', fontsize=16, fontweight='bold', pad=20)
plt.tight_layout()
plt.savefig('../visualisations/05_entity_types.png', dpi=150, bbox_inches='tight', facecolor=BG)
plt.show()

print('\n--- DATA GOVERNANCE INSIGHT ---')
ba_pct = entity_counts.get('Business Associate', 0) / entity_counts.sum() * 100
print(f'Business Associates account for {ba_pct:.1f}% of breaches.')
print('This highlights the critical importance of third-party data governance — ')
print('organisations must ensure their vendors and partners maintain equivalent data protection standards.')

## 7. Key Findings & Data Governance Implications

### Summary Statistics

In [ ]:
print('=' * 60)
print('HEALTHCARE DATA BREACH ANALYSIS — KEY FINDINGS')
print('=' * 60)
print(f'\nTotal breaches analysed: {len(df):,}')
print(f'Total individuals affected: {df["Individuals_Affected"].sum():,.0f}')
print(f'Average breach size: {df["Individuals_Affected"].mean():,.0f} individuals')
print(f'Median breach size: {df["Individuals_Affected"].median():,.0f} individuals')
print(f'Largest single breach: {df["Individuals_Affected"].max():,.0f} individuals')
print(f'\nBreaches have increased {((yearly.iloc[-1] / yearly.iloc[0]) - 1) * 100:.0f}% from {yearly.index[0]} to {yearly.index[-1]}')
print(f'Hacking now accounts for {hack_pct.iloc[-1]:.0f}% of all breaches (up from {hack_pct.iloc[0]:.0f}%)')
print(f'Median time to report: {median_delay:.0f} days')
print(f'{over_60:.1f}% of breaches exceed the 60-day HIPAA reporting window')
print(f'\n{"=" * 60}')
print('DATA GOVERNANCE RECOMMENDATIONS')
print('=' * 60)
print('\n1. IMPLEMENT CONTINUOUS MONITORING: The shift toward hacking')
print('   requires real-time data access monitoring, not periodic audits.')
print('\n2. STRENGTHEN THIRD-PARTY GOVERNANCE: Business associates account')
print(f'   for {ba_pct:.0f}% of breaches. Vendor data governance assessments')
print('   must be mandatory and regularly reviewed.')
print('\n3. REDUCE DETECTION TIMELINES: Organisations must invest in')
print('   automated breach detection through data access pattern monitoring.')
print('\n4. ENFORCE LEAST-PRIVILEGE ACCESS: Role-based access controls')
print('   at the data layer must be standard, not optional.')
print('\n5. DATA LINEAGE FOR INCIDENT RESPONSE: Organisations with')
print('   complete data lineage can identify breach scope in hours,')
print('   not months. Governance-embedded pipelines are the solution.')

---
**Analysis by Olamide Bakare** | Data Engineer & Data Governance Specialist

[LinkedIn](https://www.linkedin.com/in/olamide-bakare/) | [GitHub](https://github.com/olamidebakare) | [Medium](#)

*This analysis uses data modelled on publicly reported healthcare breaches from the HHS Office for Civil Rights Breach Portal. The dataset structure mirrors the real OCR data schema.*